# log-back — worked example 3: log_back at tiny x matches autograd exactly

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `log-back`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

log_back adds no epsilon and does no clamping: it returns `grad_out / x` faithfully, so for very small `x` it reproduces the `1/x` blow-up exactly as `torch.autograd` does. The implementation must not 'protect' against this — the blow-up is the correct mathematical answer.

## Worked solution

We stress the backward rule at small inputs and audit it element by element.

1. **No safety net.** `log_back` is literally `grad_out / x` — no `+ eps`, no `clamp_min`, no `where`. Any such guard would make us disagree with autograd.
2. **Per-element audit.** For each input we compute the hand value on a 1-element slice and the autograd value by differentiating `(log(x_i) * grad_out_i).sum()`.
3. **Tiny x.** At `x = 1e-30` the gradient is a huge finite number (`grad_out / 1e-30`); both hand and autograd produce the same large value.
4. **Comparison.** We collect `(hand, autograd)` pairs and confirm they match.

The demo prints the pairs for a few magnitudes and a final all-match flag.

In [ ]:
import torch as t

t.manual_seed(2)

def log_back(grad_out, out, x):
    return grad_out / x

x_values = t.tensor([1e-30, 1e-3, 1.0, 50.0])
grad_out_values = t.tensor([2.0, 2.0, 2.0, 2.0])

pairs = []
for i in range(x_values.numel()):
    xs = x_values[i:i+1]
    gs = grad_out_values[i:i+1]
    hand = log_back(gs, None, xs).item()
    xv = xs.clone().detach().requires_grad_(True)
    (t.log(xv) * gs).sum().backward()
    pairs.append((hand, xv.grad.item()))

print('pairs:', [(round(h, 6), round(a, 6)) for h, a in pairs])
print('all match:', all(abs(h - a) <= 1e-9 * max(1.0, abs(a)) for h, a in pairs))